# 01 - Propostas Comerciais
> Redacao de propostas tecnicas e orcamentos para clientes

**Modulo:** `14_enfitec_gestao` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

## System Prompt: Consultor Comercial de EJ

Vamos criar um **system prompt** que transforma o Claude em um consultor comercial
especializado em empresas juniores de engenharia. Isso ajusta tom, vocabulario e
estrutura das respostas para o contexto da Enfitec.

In [ ]:
SYSTEM_COMERCIAL = """
Voce e um consultor comercial senior da Enfitec, empresa junior de Engenharia
Fisica da UFRGS. Suas responsabilidades:

- Redigir propostas tecnicas claras e profissionais
- Estimar custos de projetos de engenharia (hardware, software, testes)
- Comunicar-se com clientes de forma cordial e tecnica
- Destacar os diferenciais da Enfitec: base cientifica solida, acesso a
  laboratorios da UFRGS, equipe multidisciplinar em fisica e engenharia
- Usar linguagem formal brasileira, sem girias
- Estruturar documentos com secoes claras e numeradas

Sempre responda em portugues brasileiro.
"""

print(SYSTEM_COMERCIAL)

In [ ]:
# Comparando respostas COM e SEM system prompt
pergunta = "Apresente a Enfitec para um potencial cliente do setor industrial."

print("=== SEM System Prompt ===")
print(ask(pergunta))
print()
print("=== COM System Prompt ===")
print(ask(pergunta, system=SYSTEM_COMERCIAL))

## Gerando Proposta Tecnica

A partir de uma descricao breve da necessidade do cliente, o Claude gera
uma proposta tecnica completa com escopo, entregaveis, cronograma e metodologia.

In [ ]:
briefing_cliente = """
O cliente e uma fabrica de autopecas que precisa de um sistema de monitoramento
de vibracoes em suas linhas de producao. Eles querem detectar anomalias em
maquinas rotativas (tornos e fresadoras) antes que ocorram falhas.
Orcamento estimado: R$ 15.000 a R$ 25.000.
Prazo desejado: 3 meses.
"""

prompt_proposta = f"""
Com base no seguinte briefing do cliente, redija uma proposta tecnica completa.

BRIEFING:
{briefing_cliente}

A proposta deve conter:
1. Titulo e identificacao do projeto
2. Escopo do trabalho (o que esta incluido e o que nao esta)
3. Entregaveis (lista detalhada)
4. Metodologia de trabalho
5. Cronograma com marcos (milestones)
6. Premissas e restricoes
7. Validade da proposta

Use formatacao Markdown.
"""

proposta = ask(prompt_proposta, system=SYSTEM_COMERCIAL, model=SONNET, max_tokens=2048)
print(proposta)

## Estimativa de Custos

Claude ajuda a criar uma **estrutura de custos** detalhada, incluindo
BOM (Bill of Materials), horas-pessoa por funcao, overhead e margem de lucro.

In [ ]:
prompt_custos = f"""
Para o projeto descrito abaixo, gere uma estimativa de custos detalhada
em formato de dicionario Python.

PROJETO:
{briefing_cliente}

O dicionario deve ter a seguinte estrutura:

custos = {{
    'materiais': [
        {{'item': '...', 'qtd': N, 'preco_unit': X.XX, 'total': Y.YY}},
    ],
    'horas_pessoa': [
        {{'funcao': '...', 'horas': N, 'valor_hora': X.XX, 'total': Y.YY}},
    ],
    'overhead_pct': 15,
    'margem_lucro_pct': 20,
}}

Inclua componentes reais (acelerometros, microcontroladores, PCB, etc.)
e funcoes reais (engenheiro de hardware, desenvolvedor, testador).
Retorne APENAS o codigo Python, sem explicacoes.
"""

codigo_custos = ask(prompt_custos, system=SYSTEM_COMERCIAL, model=SONNET, max_tokens=2048)
print(codigo_custos)

In [ ]:
# Exemplo de como processar a estrutura de custos
# (descomente e ajuste apos rodar a celula acima)

# exec(codigo_custos)  # cuidado: apenas para demonstracao

# Exemplo estatico para demonstracao:
custos = {
    'materiais': [
        {'item': 'Acelerometro ADXL345', 'qtd': 6, 'preco_unit': 45.00, 'total': 270.00},
        {'item': 'ESP32 DevKit', 'qtd': 3, 'preco_unit': 55.00, 'total': 165.00},
        {'item': 'PCB customizada', 'qtd': 3, 'preco_unit': 120.00, 'total': 360.00},
        {'item': 'Gabinete IP65', 'qtd': 3, 'preco_unit': 85.00, 'total': 255.00},
        {'item': 'Fonte 5V/2A', 'qtd': 3, 'preco_unit': 35.00, 'total': 105.00},
        {'item': 'Cabos e conectores', 'qtd': 1, 'preco_unit': 200.00, 'total': 200.00},
    ],
    'horas_pessoa': [
        {'funcao': 'Eng. Hardware', 'horas': 80, 'valor_hora': 35.00, 'total': 2800.00},
        {'funcao': 'Desenvolvedor SW', 'horas': 100, 'valor_hora': 35.00, 'total': 3500.00},
        {'funcao': 'Testador/QA', 'horas': 40, 'valor_hora': 30.00, 'total': 1200.00},
        {'funcao': 'Gerente de Projeto', 'horas': 30, 'valor_hora': 40.00, 'total': 1200.00},
    ],
    'overhead_pct': 15,
    'margem_lucro_pct': 20,
}

# Calculo dos totais
total_materiais = sum(m['total'] for m in custos['materiais'])
total_horas = sum(h['total'] for h in custos['horas_pessoa'])
subtotal = total_materiais + total_horas
overhead = subtotal * custos['overhead_pct'] / 100
margem = (subtotal + overhead) * custos['margem_lucro_pct'] / 100
total_final = subtotal + overhead + margem

print(f"Materiais:      R$ {total_materiais:,.2f}")
print(f"Horas-pessoa:   R$ {total_horas:,.2f}")
print(f"Subtotal:       R$ {subtotal:,.2f}")
print(f"Overhead (15%): R$ {overhead:,.2f}")
print(f"Margem (20%):   R$ {margem:,.2f}")
print(f"TOTAL FINAL:    R$ {total_final:,.2f}")

## Carta de Apresentacao

Uma carta de apresentacao profissional que acompanha a proposta,
destacando os diferenciais da Enfitec como EJ da UFRGS.

In [ ]:
prompt_carta = """
Redija uma carta de apresentacao profissional para acompanhar uma proposta
tecnica da Enfitec. A carta deve:

1. Apresentar a Enfitec como empresa junior de Engenharia Fisica da UFRGS
2. Destacar diferenciais:
   - Acesso a laboratorios e equipamentos da universidade
   - Equipe com formacao em fisica aplicada e engenharia
   - Orientacao de professores doutores
   - Custo competitivo com qualidade academica
   - Experiencia em instrumentacao e automacao
3. Mencionar areas de atuacao: sensoriamento, automacao, analise de dados,
   desenvolvimento de prototipos
4. Incluir chamada para acao (agendar reuniao)

Tom: profissional, cordial, confiante sem ser arrogante.
Formato: carta formal com cabecalho, saudacao, corpo, despedida.
"""

carta = ask(prompt_carta, system=SYSTEM_COMERCIAL, model=SONNET, max_tokens=1500)
print(carta)

## Revisao e Ajuste de Propostas

Envie um texto de proposta existente para Claude revisar: tom,
completude, clareza e linguagem profissional.

In [ ]:
proposta_rascunho = """
Oi, tudo bem?

A gente da Enfitec pode fazer um sisteminha de monitoramento pra voces.
Basicamente vai ter uns sensores nas maquinas que medem vibracao e manda
os dados pra um computador. Dai a gente faz um programinha que mostra
se a maquina ta boa ou nao.

O preco fica mais ou menos uns 20 mil, da pra negociar.
Demora uns 3 meses mais ou menos.

Qualquer coisa e so chamar!
Valeu!
"""

prompt_revisao = f"""
Revise a proposta abaixo considerando os seguintes criterios:
1. Tom e formalidade (adequado para comunicacao B2B?)
2. Completude (faltam informacoes importantes?)
3. Clareza tecnica (os termos estao corretos?)
4. Profissionalismo (a proposta transmite credibilidade?)

Para cada criterio, de uma nota de 1-5 e justifique.
Depois, reescreva a proposta de forma profissional.

PROPOSTA ORIGINAL:
{proposta_rascunho}
"""

revisao = ask(prompt_revisao, system=SYSTEM_COMERCIAL, model=SONNET, max_tokens=2048)
print(revisao)

## Template de Email para Clientes

Gerando templates de email para diferentes momentos do relacionamento
com o cliente: contato inicial, follow-up, entrega e feedback.

In [ ]:
tipos_email = [
    ("contato_inicial",
     "Primeiro contato apos o cliente demonstrar interesse via site ou evento"),
    ("follow_up",
     "Acompanhamento 5 dias apos envio de proposta sem resposta"),
    ("notificacao_entrega",
     "Informar que o projeto foi concluido e agendar entrega/demonstracao"),
    ("pedido_feedback",
     "Solicitar avaliacao do servico 2 semanas apos entrega"),
]

for tipo, descricao in tipos_email:
    prompt = f"""
Gere um template de email profissional para o seguinte cenario:
Tipo: {tipo}
Contexto: {descricao}

O email deve:
- Ter assunto atrativo
- Ser conciso (maximo 150 palavras no corpo)
- Ter tom profissional mas cordial
- Incluir placeholders como [NOME_CLIENTE], [NOME_PROJETO], etc.
- Terminar com chamada para acao clara
"""
    resultado = ask(prompt, system=SYSTEM_COMERCIAL, max_tokens=800)
    print(f"\n{'='*60}")
    print(f"TEMPLATE: {tipo.upper()}")
    print(f"{'='*60}")
    print(resultado)

---

## Exercicios

1. **System Prompt Personalizado**: Modifique o `SYSTEM_COMERCIAL` para incluir
   informacoes especificas sobre um projeto real da Enfitec. Compare os resultados.

2. **Proposta para Outro Setor**: Use o fluxo de geracao de proposta para um
   cliente do setor agricola (monitoramento de solo) ou saude (instrumentacao
   hospitalar).

3. **Tabela de Custos Automatica**: Escreva uma funcao que recebe o dicionario
   de custos e gera uma tabela formatada em Markdown automaticamente.

4. **Pipeline Completo**: Crie um fluxo que recebe apenas o briefing do cliente
   e gera automaticamente: proposta, custos, carta e email de envio.

5. **Comparacao de Modelos**: Gere a mesma proposta com Haiku e Sonnet.
   Compare qualidade, detalhe e tempo de resposta.

## Proximos Passos

- **Proximo notebook**: `02_blog_marketing.ipynb` - Criacao de conteudo para
  blog e estrategia de marketing digital
- **Integracao**: Conectar a geracao de propostas com templates em LaTeX ou
  Google Docs para formatacao automatica
- **CRM**: Integrar com ferramentas de CRM para rastrear propostas enviadas
- **Metricas**: Acompanhar taxa de conversao de propostas geradas com IA

---
*Notebook criado para o curso Claude Architect - Modulo 14: Enfitec Gestao*